In [ ]:
# Importing libraries
import hashlib
import geopandas as gpd
import pandas as pd
from google.colab import files
import math
import time
import statistics
from datetime import datetime
import json
import re
from geopy.distance import geodesic

In [ ]:
def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

raw_path = "/content/PublicAccessAEDs.geojson"  # adjust to uploaded filename/location
print(sha256_of_file(raw_path))

e2ef793ffd0fd2dbe99ffdcfb21b38154c81fd0685d1f0fcc5b75a6d57205c02


### Reading the Dataset for insights

In [ ]:
gdf = gpd.read_file(raw_path)
print(gdf.shape)
print(gdf.columns.tolist())
gdf.head()

(9644, 17)
['OBJECTID', 'AED_ID', 'OPERATING_HOURS', 'HOUSE_NUMBER', 'ROAD_NAME', 'BUILDING_NAME', 'UNIT_NUMBER', 'POSTAL_CODE', 'AED_LOCATION_DESCRIPTION', 'AED_LOCATION_FLOOR_LEVEL', 'LATITUDE', 'LONGITUDE', 'XVAL', 'YVAL', 'INC_CRC', 'FMEL_UPD_D', 'geometry']


,OBJECTID,AED_ID,OPERATING_HOURS,HOUSE_NUMBER,ROAD_NAME,BUILDING_NAME,UNIT_NUMBER,POSTAL_CODE,AED_LOCATION_DESCRIPTION,AED_LOCATION_FLOOR_LEVEL,LATITUDE,LONGITUDE,XVAL,YVAL,INC_CRC,FMEL_UPD_D,geometry
0,18662,098136-004,Mon - Sun 00:00-23:59;,22,Sentosa Gateway,SEA Aquarium,None,98136,Level B1M Open Ocean Habitat (Highest Tier Of Viewing Gallery),B1M,1.258654,103.818390,26338.58154,26801.16493,0CD98DA545DAAB9D,20200210102016,POINT (103.81839 1.25865)
1,18663,098136-006,Mon - Sun 00:00-23:59;,22,Sentosa Gateway,SEA Aquarium,None,98136,Level B1 Freshwater Lakes (Left Side Of Exhibit),B1,1.258555,103.818748,26378.48903,26790.22402,50C661B41C959E4A,20200210102016,POINT (103.81875 1.25856)
2,18664,098137-001,Mon - Sun 00:00-23:59;,24,Sentosa Gateway,SEA Aquarium,None,98137,"Level B1 Near SEA Aquarium Entrance Or Exit, Opposite Passenger Lift",B1,1.258804,103.820358,26557.64860,26817.74614,613A270C404E6AE4,20200210102016,POINT (103.82036 1.2588)
3,18665,098138-001,Mon - Sun 00:00-23:59;,26,Sentosa Gateway,Resorts World Sentosa,None,98138,Level 1 Beside Hard Rock Café (Beside Staircase Leading To Hard Rock Hotel),1,1.257367,103.820251,26545.76045,26658.85586,3FBC0007241B3217,20200210102016,POINT (103.82025 1.25737)
4,18666,098138-002,Mon - Sun 00:00-23:59;,26,Sentosa Gateway,Resorts World Sentosa,None,98138,Level B1 VIP Car Park Beside Crash Barrier,B1,1.256337,103.820343,26555.94905,26544.96726,CE8E0C8F7C49B7AE,20200210102016,POINT (103.82034 1.25634)


In [ ]:
gdf.isnull().sum()

,0
OBJECTID,0
AED_ID,0
OPERATING_HOURS,7
HOUSE_NUMBER,97
ROAD_NAME,0
BUILDING_NAME,5265
UNIT_NUMBER,9301
POSTAL_CODE,0
AED_LOCATION_DESCRIPTION,0
AED_LOCATION_FLOOR_LEVEL,205


In [ ]:
print(gdf.shape[0])

9644


In [ ]:
print(gdf['OPERATING_HOURS'].value_counts().head(30))
print(gdf['OPERATING_HOURS'].isna().sum())

OPERATING_HOURS
Mon - Sun 00:00-23:59;                                                                                                                                                                                     6570
Mon - Sun 07:00-17:00;                                                                                                                                                                                     1417
Mon - Fri 08:30-18:00; Sat - Sun Closed;                                                                                                                                                                    369
Mon - Sun 10:00-23:00;                                                                                                                                                                                      167
Mon - Sun 05:30-23:59;                                                                                                                                  

In [ ]:
# all the raw strings that appear in this column, exactly as written
# Including nulls in the count instead of silently dropping them

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

pattern_counts = gdf['OPERATING_HOURS'].value_counts(dropna=False).sort_values(ascending=False)
print(pattern_counts)
print(f"\nTotal distinct patterns (including null): {pattern_counts.shape[0]}")

OPERATING_HOURS
Mon - Sun 00:00-23:59;                                                                                                                                                                                                                                                                                                                                                                                                              6570
Mon - Sun 07:00-17:00;                                                                                                                                                                                                                                                                                                                                                                                                              1417
Mon - Fri 08:30-18:00; Sat - Sun Closed;                                                                                              

In [ ]:
# If ignore the Remarks noise, how many truly distinct structural patterns are there and how many rows even have a Remarks clause:
# Stripping out everything from "Remarks" onward, to see the base structural patterns
base_pattern = gdf['OPERATING_HOURS'].str.split('Remarks:').str[0].str.strip()
base_counts = base_pattern.value_counts(dropna=False)
print(f"Distinct BASE patterns (ignoring remarks): {base_counts.shape[0]}")
print(base_counts)

# Separately, just check how many rows even HAVE a Remarks clause
has_remarks = gdf['OPERATING_HOURS'].str.contains('Remarks:', na=False)
print(f"\nRows with a Remarks clause: {has_remarks.sum()} out of {gdf.shape[0]}")

Distinct BASE patterns (ignoring remarks): 122
OPERATING_HOURS
Mon - Sun 00:00-23:59;                                                                                                    6572
Mon - Sun 07:00-17:00;                                                                                                    1417
Mon - Fri 08:30-18:00; Sat - Sun Closed;                                                                                   369
Mon - Sun 05:30-23:59;                                                                                                     168
Mon - Sun 10:00-23:00;                                                                                                     167
Mon - Sun 09:00-22:00;                                                                                                     162
Mon - Sun Closed;                                                                                                          111
Mon - Sun 07:00-22:00;                          

In [ ]:
DAY_ORDER = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']

def expand_day_range(start_day, end_day):
    """Turn 'MON'-'FRI' into ['MON','TUE','WED','THU','FRI']"""
    si, ei = DAY_ORDER.index(start_day), DAY_ORDER.index(end_day)
    if si <= ei:
        return DAY_ORDER[si:ei+1]
    return DAY_ORDER[si:] + DAY_ORDER[:ei+1]  # handles wraparound, e.g. Sat-Mon

# Matches one segment like "MON - FRI 08:30-18:00" or "SAT - SUN CLOSED"
SEGMENT_RE = re.compile(
    r'^([A-Z]{3})(?:\s*-\s*([A-Z]{3}))?\s+(CLOSED|\d{2}:\d{2}\s*-\s*\d{2}:\d{2})$'
)

# classify what kind of remark we're dealing with, so we can tell
# "safe direction" (midnight rollover) apart from "dangerous direction"
# (mid-day gap) and "unschedulable" (conditional access)
def classify_remark_risk(raw_text):
    if "Remarks:" not in raw_text:
        return None
    remark_part = raw_text.split("Remarks:")[1]
    if "Closed from" in remark_part:
        return "mid_day_gap"         # dangerous! must not claim open here
    if "Depending on" in remark_part or "Please Call" in remark_part:
        return "conditional_access"  # can't be scheduled at all
    if "Closes at" in remark_part:
        return "midnight_rollover"   # safe direction, documented limitation
    return "other_remark"


def parse_operating_hours(raw_text):
    if raw_text is None or (isinstance(raw_text, float) and pd.isna(raw_text)) or str(raw_text).strip() == "":
        return {"status": "unknown", "confidence": "none", "schedule": {}, "raw": raw_text}

    text = str(raw_text).strip()

    has_remarks = "Remarks:" in text
    base_text = text.split("Remarks:")[0].strip().upper()

    segments = [s.strip() for s in base_text.split(';') if s.strip()]

    schedule = {}
    parse_failed = False

    for seg in segments:
        match = SEGMENT_RE.match(seg)
        if not match:
            parse_failed = True
            continue
        start_day, end_day, hours_part = match.groups()
        days = expand_day_range(start_day, end_day) if end_day else [start_day]

        if hours_part == "CLOSED":
            for d in days:
                schedule[d] = None
        else:
            open_t, close_t = [t.strip() for t in hours_part.split('-')]
            for d in days:
                schedule[d] = (open_t, close_t)

    if not schedule or parse_failed:
        return {"status": "ambiguous", "confidence": "low", "schedule": schedule, "raw": raw_text}

    # check remark type BEFORE deciding confidence/status.
    # This block replaces the old single-line
    #   confidence = "medium" if has_remarks else "high"
    remark_type = classify_remark_risk(text) if has_remarks else None

    if remark_type in ("mid_day_gap", "conditional_access"):
        # Base schedule parsed fine, but the remark describes something
        # our schedule model can't safely represent (an unstated closure
        # gap, or fully conditional access). Don't claim "open" here.
        return {"status": "ambiguous", "confidence": "low",
                "reason": remark_type, "schedule": schedule, "raw": raw_text}

    confidence = "medium" if has_remarks else "high"
    return {"status": "parsed", "confidence": confidence, "schedule": schedule,
            "has_remarks": has_remarks, "remark_type": remark_type, "raw": raw_text}

In [ ]:
gdf['parsed_hours'] = gdf['OPERATING_HOURS'].apply(parse_operating_hours)
status_counts = gdf['parsed_hours'].apply(lambda x: x['status']).value_counts()
print(status_counts)

parsed_hours
parsed       9630
unknown         7
ambiguous       7
Name: count, dtype: int64


In [ ]:
confidence_counts = gdf['parsed_hours'].apply(lambda x: x.get('confidence')).value_counts()
print(confidence_counts)

parsed_hours
high      9575
medium      55
none         7
low          7
Name: count, dtype: int64


In [ ]:
sample = gdf[gdf['parsed_hours'].apply(lambda x: x.get('has_remarks', False))][['OPERATING_HOURS', 'parsed_hours']].head(5)
for _, row in sample.iterrows():
    print("RAW:", row['OPERATING_HOURS'])
    print("PARSED:", row['parsed_hours'])
    print()

RAW: Mon - Sun 06:00-23:59; Remarks: Mon: Closes at 3:00 AM, Tue: Closes at 3:00 AM, Wed: Closes at 3:00 AM, Thu: Closes at 3:00 AM, Fri: Closes at 3:00 AM, Sat: Closes at 3:00 AM, Sun: Closes at 3:00 AM;
PARSED: {'status': 'parsed', 'confidence': 'medium', 'schedule': {'MON': ('06:00', '23:59'), 'TUE': ('06:00', '23:59'), 'WED': ('06:00', '23:59'), 'THU': ('06:00', '23:59'), 'FRI': ('06:00', '23:59'), 'SAT': ('06:00', '23:59'), 'SUN': ('06:00', '23:59')}, 'has_remarks': True, 'remark_type': 'midnight_rollover', 'raw': 'Mon - Sun 06:00-23:59; Remarks: Mon: Closes at 3:00 AM, Tue: Closes at 3:00 AM, Wed: Closes at 3:00 AM, Thu: Closes at 3:00 AM, Fri: Closes at 3:00 AM, Sat: Closes at 3:00 AM, Sun: Closes at 3:00 AM;'}

RAW: Mon - Sun 06:00-23:59; Remarks: Mon: Closes at 3:00 AM, Tue: Closes at 3:00 AM, Wed: Closes at 3:00 AM, Thu: Closes at 3:00 AM, Fri: Closes at 3:00 AM, Sat: Closes at 3:00 AM, Sun: Closes at 3:00 AM;
PARSED: {'status': 'parsed', 'confidence': 'medium', 'schedule': {

In [ ]:
remarks_only = gdf[gdf['parsed_hours'].apply(lambda x: x.get('has_remarks', False))]['OPERATING_HOURS'].str.split('Remarks:').str[1]
print(remarks_only.value_counts().head(20))

OPERATING_HOURS
Mon: Closes at 1:30 AM, Tue: Closes at 1:30 AM, Wed: Closes at 1:30 AM, Thu: Closes at 1:30 AM, Fri: Closes at 1:30 AM, Sat: Closes at 1:30 AM, Sun: Closes at 1:30 AM;      14
Mon: Closes at 1:00 AM, Tue: Closes at 1:00 AM, Wed: Closes at 1:00 AM, Thu: Closes at 1:00 AM, Fri: Closes at 1:00 AM, Sat: Closes at 1:00 AM, Sun: Closes at 1:00 AM;       9
Mon: Closes at 3:00 AM, Tue: Closes at 3:00 AM, Wed: Closes at 3:00 AM, Thu: Closes at 3:00 AM, Fri: Closes at 3:00 AM, Sat: Closes at 3:00 AM, Sun: Closes at 3:00 AM;       7
Fri: Closes at 1:00 AM, Sat: Closes at 1:00 AM;                                                                                                                               6
Fri: Closes at 2:00 AM, Sat: Closes at 2:00 AM;                                                                                                                               6
Mon: Closes at 2:00 AM, Tue: Closes at 2:00 AM, Wed: Closes at 2:00 AM, Thu: Closes at 2:00 AM, Fri: Clo

In [ ]:
gdf.to_file("/content/aed_parsed.geojson", driver="GeoJSON")
print(sha256_of_file("/content/aed_parsed.geojson"))

d88b165d5e22f4ed3d4a69f67f4db31e5c5e5bd5345f42408792ede1d50b8783


In [ ]:
scenarios=[
  {
    "scenario_id": "S01",
    "location_name": "Bugis Junction",
    "day_of_week": "Tuesday",
    "time": "15:00",
    "lat": 1.2995890796324496,
    "lng": 103.85552397170694,
    "notes": "weekday daytime, dense commercial area"
  },
  {
    "scenario_id": "S02",
    "location_name": "Toa Payoh HDB Hub",
    "day_of_week": "Sunday",
    "time": "23:00",
    "lat": 1.3328409657023292,
    "lng": 103.84806838269759,
    "notes": "weekend nighttime, residential/sparse area"
  },
  {
    "scenario_id": "S03",
    "location_name": "Orchard Road (ION Orchard)",
    "day_of_week": "Saturday",
    "time": "13:00",
    "lat": 1.3047048950723552,
    "lng": 103.83213542509972,
    "notes": "weekend midday, dense shopping district"
  },
  {
    "scenario_id": "S04",
    "location_name": "Pioneer Walk",
    "day_of_week": "Saturday",
    "time": "20:00",
    "lat": 1.3193796705023573,
    "lng": 103.69495876550073,
    "notes": "weekend evening, industrial/western area"
  },
  {
    "scenario_id": "S05",
    "location_name": "Jurong Point",
    "day_of_week": "Wednesday",
    "time": "11:00",
    "lat": 1.3396,
    "lng": 103.7066,
    "notes": "weekday late morning, dense mall area"
  },
  {
    "scenario_id": "S06",
    "location_name": "Yishun Ring Road (residential)",
    "day_of_week": "Monday",
    "time": "06:30",
    "lat": 1.4263975894460061,
    "lng": 103.8321241520087,
    "notes": "weekday early morning, sparse residential, tests office-hours edge case"
  },
  {
    "scenario_id": "S07",
    "location_name": "Tampines Mall",
    "day_of_week": "Friday",
    "time": "19:00",
    "lat": 1.3526,
    "lng": 103.9447,
    "notes": "weekday evening, dense eastern hub"
  },
  {
    "scenario_id": "S08",
    "location_name": "Chinatown (Pagoda Street)",
    "day_of_week": "Thursday",
    "time": "22:30",
    "lat": 1.2836,
    "lng": 103.8441,
    "notes": "weekday late night, mixed heritage/commercial, tests closed-shop scenarios"
  },
  {
    "scenario_id": "S09",
    "location_name": "Punggol Waterway Point",
    "day_of_week": "Sunday",
    "time": "10:00",
    "lat": 1.4065,
    "lng": 103.9020,
    "notes": "weekend morning, newer sparse-AED-coverage area, tests low-candidate-density edge case"
  },
  {
    "scenario_id": "S10",
    "location_name": "Bishan-Ang Mo Kio Park",
    "day_of_week": "Wednesday",
    "time": "17:30",
    "lat": 1.3636,
    "lng": 103.8435,
    "notes": "weekday late afternoon, park/outdoor area, tests ambiguous-location-description edge case"
  },
  {
    "scenario_id": "S11",
    "location_name": "Marina Bay Sands",
    "day_of_week": "Friday",
    "time": "21:00",
    "lat": 1.2834,
    "lng": 103.8607,
    "notes": "weekday night, dense tourist/commercial hotspot"
  },
  {
      "scenario_id": "S12",
      "location_name": "Sentosa Beach Station",
      "day_of_week": "Saturday",
      "time": "14:00",
      "lat": 1.2494,
      "lng": 103.8303,
      "notes": "weekend afternoon, leisure/island area, tests lower AED density"
  },
  {
      "scenario_id": "S13",
      "location_name": "Woodlands Checkpoint",
      "day_of_week": "Monday",
      "time": "07:30",
      "lat": 1.4382,
      "lng": 103.7691,
      "notes": "weekday early morning, border/industrial area, tests long-distance case"
  },
  {
      "scenario_id": "S14",
      "location_name": "Clementi Town Centre",
      "day_of_week": "Tuesday",
      "time": "18:30",
      "lat": 1.3151,
      "lng": 103.7649,
      "notes": "weekday evening, mixed residential/commercial density"
  },
  {
      "scenario_id": "S15",
      "location_name": "Serangoon NEX Mall",
      "day_of_week": "Sunday",
      "time": "16:00",
      "lat": 1.3506,
      "lng": 103.8721,
      "notes": "weekend evening, dense mall/transport hub"
  },
  {
      "scenario_id": "S16",
      "location_name": "Bukit Timah Nature Reserve",
      "day_of_week": "Saturday",
      "time": "08:00",
      "lat": 1.3541,
      "lng": 103.7764,
      "notes": "weekend early morning, nature/sparse area, tests no-qualifying-AED case"
  },
  {
      "scenario_id": "S17",
      "location_name": "Queenstown (Dawson)",
      "day_of_week": "Wednesday",
      "time": "12:30",
      "lat": 1.2977,
      "lng": 103.8060,
      "notes": "weekday midday, mixed residential area"
  },
  {
      "scenario_id": "S18",
      "location_name": "Novena Square",
      "day_of_week": "Thursday",
      "time": "23:30",
      "lat": 1.3204,
      "lng": 103.8437,
      "notes": "weekday late night, tests closed-shop/uncertain-hours edge case"
  },
  {
      "scenario_id": "S19",
      "location_name": "Pasir Ris Park",
      "day_of_week": "Sunday",
      "time": "06:00",
      "lat": 1.3816,
      "lng": 103.9556,
      "notes": "weekend early morning, coastal/park area, tests low-confidence + far-distance combo"
  },
  {
      "scenario_id": "S20",
      "location_name": "Boon Lay (Nanyang area)",
      "day_of_week": "Friday",
      "time": "22:00",
      "lat": 1.3483,
      "lng": 103.6831,
      "notes": "weekday late night, industrial/university sparse area, tests ambiguous-location-description case"
  }
]

In [ ]:
with open("/content/cached_scenarios.json", "w") as f:
    json.dump(scenarios, f, indent=2)

In [ ]:
# Setup: loading scenarios and parsed AED data
# If scenarios aren't already in memory, load the file saved earlier
with open("/content/cached_scenarios.json", "r") as f:
    scenarios = json.load(f)

# gdf should already be the parsed dataframe from Step 5
# Make sure it has 'LATITUDE' and 'LONGITUDE' as usable floats
aed_records = gdf[['AED_ID', 'LATITUDE', 'LONGITUDE', 'BUILDING_NAME',
                    'AED_LOCATION_DESCRIPTION', 'parsed_hours']].to_dict('records')

# For each scenario, find the N nearest AEDs by straight-line distance
N_CANDIDATES = 10

def get_candidates(origin_lat, origin_lng, aed_records, n=N_CANDIDATES):
    scored = []
    for aed in aed_records:
        dist_m = geodesic((origin_lat, origin_lng),
                           (aed['LATITUDE'], aed['LONGITUDE'])).meters
        scored.append({**aed, "straight_line_distance_m": round(dist_m, 1)})
    scored.sort(key=lambda x: x['straight_line_distance_m'])
    return scored[:n]

# Walking distance estimated via standard urban pedestrian detour factor,
# applied to straight-line distance i.e., no external routing API required.
# 1.35x is a commonly cited detour multiplier in pedestrian-accessibility
# research for typical urban street grids.
DETOUR_FACTOR = 1.35

def estimate_walking_distance(straight_line_distance_m, detour_factor=DETOUR_FACTOR):
    return round(straight_line_distance_m * detour_factor, 1)

# Run it all, build the combined cache, and save
cached_routes = []

for scenario in scenarios:
    print(f"Processing {scenario['scenario_id']} - {scenario['location_name']}")
    candidates = get_candidates(scenario['lat'], scenario['lng'], aed_records)

    for cand in candidates:
        walking_dist = estimate_walking_distance(cand['straight_line_distance_m'])
        cached_routes.append({
            "scenario_id": scenario['scenario_id'],
            "aed_id": cand['AED_ID'],
            "straight_line_distance_m": cand['straight_line_distance_m'],
            "walking_distance_m": walking_dist,
            "walking_distance_method": "detour_factor_estimate_1.35x",
            "building_name": cand['BUILDING_NAME'],
            "location_description": cand['AED_LOCATION_DESCRIPTION'],
            "parsed_hours": cand['parsed_hours'],
        })

# Save the routing cache - this is what your ranking engine will read from
with open("/content/cached_routes.json", "w") as f:
    json.dump(cached_routes, f, indent=2)

print(f"\nDone. Cached {len(cached_routes)} scenario-candidate pairs "
      f"across {len(scenarios)} scenarios.")

Processing S01 — Bugis Junction
Processing S02 — Toa Payoh HDB Hub
Processing S03 — Orchard Road (ION Orchard)
Processing S04 — Pioneer Walk
Processing S05 — Jurong Point
Processing S06 — Yishun Ring Road (residential)
Processing S07 — Tampines Mall
Processing S08 — Chinatown (Pagoda Street)
Processing S09 — Punggol Waterway Point
Processing S10 — Bishan-Ang Mo Kio Park
Processing S11 — Marina Bay Sands
Processing S12 — Sentosa Beach Station
Processing S13 — Woodlands Checkpoint
Processing S14 — Clementi Town Centre
Processing S15 — Serangoon NEX Mall
Processing S16 — Bukit Timah Nature Reserve
Processing S17 — Queenstown (Dawson)
Processing S18 — Novena Square
Processing S19 — Pasir Ris Park
Processing S20 — Boon Lay (Nanyang area)

Done. Cached 200 scenario-candidate pairs across 20 scenarios.


In [ ]:
print(json.dumps(cached_routes[0], indent=2))

{
  "scenario_id": "S01",
  "aed_id": "188022-001",
  "straight_line_distance_m": 71.5,
  "walking_distance_m": 96.5,
  "walking_distance_method": "detour_factor_estimate_1.35x",
  "building_name": "Bugis MRT Station",
  "location_description": "Near Passenger Service Centre (EWL)",
  "parsed_hours": {
    "status": "parsed",
    "confidence": "high",
    "schedule": {
      "MON": [
        "05:30",
        "23:59"
      ],
      "TUE": [
        "05:30",
        "23:59"
      ],
      "WED": [
        "05:30",
        "23:59"
      ],
      "THU": [
        "05:30",
        "23:59"
      ],
      "FRI": [
        "05:30",
        "23:59"
      ],
      "SAT": [
        "05:30",
        "23:59"
      ],
      "SUN": [
        "05:30",
        "23:59"
      ]
    },
    "has_remarks": false,
    "remark_type": null,
    "raw": "Mon - Sun 05:30-23:59;"
  }
}


In [ ]:
routes_df = pd.DataFrame(cached_routes)
print(routes_df['straight_line_distance_m'].describe())
print(routes_df.groupby('scenario_id')['straight_line_distance_m'].min())

count    200.000000
mean     239.639500
std      217.591521
min       15.600000
25%       87.350000
50%      160.750000
75%      303.400000
max      927.400000
Name: straight_line_distance_m, dtype: float64
scenario_id
S01     71.5
S02     50.0
S03     54.4
S04    222.8
S05     15.7
S06     77.9
S07     18.0
S08    105.6
S09     36.3
S10    157.7
S11     15.7
S12    108.4
S13    141.4
S14     35.7
S15     15.6
S16     66.3
S17    127.8
S18     18.0
S19     59.6
S20     57.6
Name: straight_line_distance_m, dtype: float64


In [ ]:
def compute_hours_score(parsed_hours, day_of_week, time_str):
    """
    Returns (hours_score 0-1, is_feasible bool).
    is_feasible=False means: don't recommend this AED for this scenario at all.
    """
    status = parsed_hours.get('status')
    confidence = parsed_hours.get('confidence')

    # Unknown or ambiguous hours -> cannot confirm accessibility -> abstain
    if status in ('unknown', 'ambiguous'):
        return 0.0, False

    schedule = parsed_hours.get('schedule', {})
    day_key = day_of_week[:3].upper()  # "Tuesday" -> "TUE"
    day_hours = schedule.get(day_key)

    if day_hours is None:  # explicitly closed that day
        return 0.0, False

    open_t, close_t = day_hours
    query_time = datetime.strptime(time_str, "%H:%M").time()
    open_time = datetime.strptime(open_t, "%H:%M").time()
    close_time = datetime.strptime(close_t, "%H:%M").time()

    is_open = open_time <= query_time <= close_time
    if not is_open:
        return 0.0, False

    # Open and confirmed -> score reflects confidence (medium confidence
    # rows still get credit, but slightly less, per our Remarks handling)
    score = 1.0 if confidence == "high" else 0.7
    return score, True


def compute_distance_score(walking_distance_m, max_distance_m=1000):
    """Closer = higher score, normalized 0-1, capped at max_distance_m."""
    capped = min(walking_distance_m, max_distance_m)
    return round(1 - (capped / max_distance_m), 3)


def rank_aeds(scenario, candidates, weights):
    """
    scenario: dict with 'day_of_week' and 'time'
    candidates: list of dicts from cached_routes.json for this scenario_id
    weights: e.g. {'distance': 0.5, 'hours': 0.5}
    """
    scored = []
    for c in candidates:
        hours_score, is_feasible = compute_hours_score(
            c['parsed_hours'], scenario['day_of_week'], scenario['time']
        )
        distance_score = compute_distance_score(c['walking_distance_m'])

        overall_score = (
            weights['distance'] * distance_score +
            weights['hours'] * hours_score
        )

        scored.append({
            **c,
            "distance_score": distance_score,
            "hours_score": hours_score,
            "is_feasible": is_feasible,
            "overall_score": overall_score,
        })

    # Only rank feasible AEDs i.e., infeasible ones (closed) don't get recommended
    feasible = [c for c in scored if c['is_feasible']]
    infeasible = [c for c in scored if not c['is_feasible']]

    ranked = sorted(feasible, key=lambda x: x['overall_score'], reverse=True)
    return ranked, infeasible  # return both i.e., infeasible list useful for your report

In [ ]:
def baseline_nearest(candidates, top_k=3):
    """
    Required baseline: nearest AED by straight-line distance, WITHOUT
    hours-awareness or learned ranking. Deliberately naive as this is
    what your ranker needs to beat.
    """
    ranked = sorted(candidates, key=lambda x: x['straight_line_distance_m'])
    return ranked[:top_k]

### Testing

In [ ]:
with open("/content/cached_routes.json") as f:
    all_routes = json.load(f)
with open("/content/cached_scenarios.json") as f:
    scenarios = json.load(f)

s01_candidates = [r for r in all_routes if r['scenario_id'] == 'S01']
s01_scenario = next(s for s in scenarios if s['scenario_id'] == 'S01')

weights = {"distance": 0.5, "hours": 0.5}
ranked, infeasible = rank_aeds(s01_scenario, s01_candidates, weights)
baseline_top3 = baseline_nearest(s01_candidates)

print("=== Your system top-3 ===")
for r in ranked[:3]:
    print(f"{r['aed_id']}: score={r['overall_score']:.3f}, "
          f"walking={r['walking_distance_m']}m, feasible={r['is_feasible']}")

print(f"\n{len(infeasible)} AEDs excluded as infeasible for this scenario")

print("\n=== Baseline top-3 ===")
for r in baseline_top3:
    print(f"{r['aed_id']}: straight_line={r['straight_line_distance_m']}m "
          f"(hours not checked)")

=== Your system top-3 ===
188022-001: score=0.952, walking=96.5m, feasible=True
188021-001: score=0.944, walking=111.6m, feasible=True
188476-001: score=0.906, walking=186.6m, feasible=True

0 AEDs excluded as infeasible for this scenario

=== Baseline top-3 ===
188022-001: straight_line=71.5m (hours not checked)
188021-001: straight_line=82.7m (hours not checked)
188476-001: straight_line=138.2m (hours not checked)


In [ ]:
s08_candidates = [r for r in all_routes if r['scenario_id'] == 'S08']
s08_scenario = next(s for s in scenarios if s['scenario_id'] == 'S08')

ranked_08, infeasible_08 = rank_aeds(s08_scenario, s08_candidates, weights)
baseline_08 = baseline_nearest(s08_candidates)

print("=== S08 Your system top-3 (should exclude closed AEDs) ===")
for r in ranked_08[:3]:
    print(f"{r['aed_id']}: score={r['overall_score']:.3f}, "
          f"walking={r['walking_distance_m']}m, feasible={r['is_feasible']}")

print(f"\n{len(infeasible_08)} AEDs excluded as infeasible")
for r in infeasible_08[:5]:
    print(f"  EXCLUDED: {r['aed_id']} — hours_score={r['hours_score']}, "
          f"raw_hours={r['parsed_hours'].get('raw', 'N/A')}")

print("\n=== S08 Baseline top-3 (ignores hours entirely) ===")
for r in baseline_08:
    print(f"{r['aed_id']}: straight_line={r['straight_line_distance_m']}m")

=== S08 Your system top-3 (should exclude closed AEDs) ===
059443-001: score=0.928, walking=142.6m, feasible=True
058945-001: score=0.895, walking=208.6m, feasible=True
059413-002: score=0.891, walking=219.4m, feasible=True

2 AEDs excluded as infeasible
  EXCLUDED: 058362-001 — hours_score=0.0, raw_hours=Mon - Sun 07:00-17:00;
  EXCLUDED: 051335-001 — hours_score=0.0, raw_hours=Mon - Fri 08:30-18:00; Sat - Sun Closed;

=== S08 Baseline top-3 (ignores hours entirely) ===
059443-001: straight_line=105.6m
058362-001: straight_line=131.6m
051335-001: straight_line=147.3m


In [ ]:
# Confirming that it is being pulled from correct cached scenarios
print(f"Total candidates: {len(s08_candidates)}, feasible: {len(ranked_08)}, infeasible: {len(infeasible_08)}")

Total candidates: 10, feasible: 8, infeasible: 2


In [ ]:
# Running both systems across all 10 scenarios, collecting everything

evaluation_rows = []
latencies = []

for scenario in scenarios:
    sid = scenario['scenario_id']
    candidates = [r for r in all_routes if r['scenario_id'] == sid]

    # Time the ranking call for p95 latency metric
    start = time.perf_counter()
    ranked, infeasible = rank_aeds(scenario, candidates, weights)
    elapsed_ms = (time.perf_counter() - start) * 1000
    latencies.append(elapsed_ms)

    baseline_top3 = baseline_nearest(candidates, top_k=3)
    system_top3 = ranked[:3]

    # infeasible_ids is only ever used for membership checks, so a set is fine here
    infeasible_ids = {c['aed_id'] for c in infeasible}

    # FIX: preserve rank order (list comprehension), don't collapse into a set
    baseline_top3_ordered = [c['aed_id'] for c in baseline_top3]
    system_top3_ordered = [c['aed_id'] for c in system_top3]

    # FIX: build this as an ordered list too, filtering baseline's ranked
    # order rather than intersecting two unordered sets
    baseline_false_accessible = [
        aid for aid in baseline_top3_ordered if aid in infeasible_ids
    ]

    evaluation_rows.append({
        "scenario_id": sid,
        "location": scenario['location_name'],
        "day_time": f"{scenario['day_of_week']} {scenario['time']}",
        "total_candidates": len(candidates),
        "feasible_count": len(ranked),
        "infeasible_count": len(infeasible),
        "system_top3": system_top3_ordered,
        "baseline_top3": baseline_top3_ordered,
        "baseline_false_accessible_count": len(baseline_false_accessible),
        "baseline_false_accessible_aeds": baseline_false_accessible,
        "latency_ms": round(elapsed_ms, 3),
    })

# Computing the 4 metrics

total_top3_slots = sum(min(3, r['feasible_count']) for r in evaluation_rows)
total_top3_filled = sum(len(r['system_top3']) for r in evaluation_rows)
top3_recall = total_top3_filled / total_top3_slots if total_top3_slots else 0

scenarios_with_baseline_error = sum(
    1 for r in evaluation_rows if r['baseline_false_accessible_count'] > 0
)
false_accessible_rate = scenarios_with_baseline_error / len(evaluation_rows)

sorted_latencies = sorted(latencies)
p95_index = int(len(sorted_latencies) * 0.95)
p95_latency = sorted_latencies[min(p95_index, len(sorted_latencies) - 1)]

system_top1_distances, baseline_top1_distances = [], []
for scenario in scenarios:
    sid = scenario['scenario_id']
    candidates = [r for r in all_routes if r['scenario_id'] == sid]
    ranked, _ = rank_aeds(scenario, candidates, weights)
    baseline_top1 = baseline_nearest(candidates, top_k=1)[0]
    if ranked:
        system_top1_distances.append(ranked[0]['walking_distance_m'])
    baseline_top1_distances.append(baseline_top1['straight_line_distance_m'])

avg_system_dist = statistics.mean(system_top1_distances)
avg_baseline_dist = statistics.mean(baseline_top1_distances)

# Print summary

print("=" * 60)
print("EVALUATION SUMMARY - 10 scenarios")
print("=" * 60)
print(f"\nPRIMARY - Top-3 feasible-AED recall: {top3_recall:.1%}")
print(f"SAFETY - Baseline false-accessible rate: {false_accessible_rate:.1%} "
      f"({scenarios_with_baseline_error}/{len(evaluation_rows)} scenarios)")
print(f"PERFORMANCE - p95 latency: {p95_latency:.2f} ms")
print(f"SUPPORTING - Avg distance: system={avg_system_dist:.1f}m "
      f"(walking-adjusted) vs baseline={avg_baseline_dist:.1f}m (straight-line)")

print("\n" + "=" * 60)
print("PER-SCENARIO DETAIL")
print("=" * 60)
for r in evaluation_rows:
    flag = " *** BASELINE FAILURE ***" if r['baseline_false_accessible_count'] > 0 else ""
    print(f"\n{r['scenario_id']} - {r['location']} ({r['day_time']}){flag}")
    print(f"  Candidates: {r['total_candidates']} total, "
          f"{r['feasible_count']} feasible, {r['infeasible_count']} infeasible")
    print(f"  System top-3 (ranked): {r['system_top3']}")
    print(f"  Baseline top-3 (ranked): {r['baseline_top3']}")
    if r['baseline_false_accessible_count'] > 0:
        print(f"  Baseline recommended CLOSED AED(s): "
              f"{r['baseline_false_accessible_aeds']}")
    print(f"  Latency: {r['latency_ms']} ms")

# Save results bundle

results_bundle = {
    "metrics": {
        "primary_top3_feasible_recall": round(top3_recall, 4),
        "safety_baseline_false_accessible_rate": round(false_accessible_rate, 4),
        "performance_p95_latency_ms": round(p95_latency, 3),
        "supporting_avg_distance_m": {
            "system_walking_adjusted": round(avg_system_dist, 1),
            "baseline_straight_line": round(avg_baseline_dist, 1),
        },
    },
    "per_scenario": evaluation_rows,
}

with open("/content/evaluation_results.json", "w") as f:
    json.dump(results_bundle, f, indent=2)

print("\n\nSaved to /content/evaluation_results.json")

EVALUATION SUMMARY - 10 scenarios

PRIMARY - Top-3 feasible-AED recall: 100.0%
SAFETY - Baseline false-accessible rate: 30.0% (6/20 scenarios)
PERFORMANCE - p95 latency: 0.38 ms
SUPPORTING - Avg distance: system=99.2m (walking-adjusted) vs baseline=72.8m (straight-line)

PER-SCENARIO DETAIL

S01 - Bugis Junction (Tuesday 15:00)
  Candidates: 10 total, 10 feasible, 0 infeasible
  System top-3 (ranked): ['188022-001', '188021-001', '188476-001']
  Baseline top-3 (ranked): ['188022-001', '188021-001', '188476-001']
  Latency: 0.382 ms

S02 - Toa Payoh HDB Hub (Sunday 23:00) *** BASELINE FAILURE ***
  Candidates: 10 total, 8 feasible, 2 infeasible
  System top-3 (ranked): ['319398-001', '310480-002', '310480-003']
  Baseline top-3 (ranked): ['310490-002', '310490-001', '319398-001']
  Baseline recommended CLOSED AED(s): ['310490-002', '310490-001']
  Latency: 0.192 ms

S03 - Orchard Road (ION Orchard) (Saturday 13:00)
  Candidates: 10 total, 10 feasible, 0 infeasible
  System top-3 (ranked

In [ ]:
# 500 time run to ensure p95 working over large number of scenarios as well
# Run the ranking function repeatedly across all 10 scenarios, purely for
# timing purposes, to get a real distribution instead of just 10 samples
timing_latencies = []

for _ in range(25):  # 25 full sweeps × 10 scenarios = 500 timed calls
    for scenario in scenarios:
        sid = scenario['scenario_id']
        candidates = [r for r in all_routes if r['scenario_id'] == sid]
        start = time.perf_counter()
        rank_aeds(scenario, candidates, weights)
        elapsed_ms = (time.perf_counter() - start) * 1000
        timing_latencies.append(elapsed_ms)

sorted_t = sorted(timing_latencies)
p95_index = int(len(sorted_t) * 0.95)
p95_robust = sorted_t[p95_index]
median_robust = statistics.median(sorted_t)

print(f"Total timed calls: {len(timing_latencies)}")
print(f"Median latency: {median_robust:.3f} ms")
print(f"p95 latency: {p95_robust:.3f} ms")
print(f"Min: {min(sorted_t):.3f} ms, Max: {max(sorted_t):.3f} ms")

Total timed calls: 500
Median latency: 0.238 ms
p95 latency: 0.445 ms
Min: 0.190 ms, Max: 1.240 ms


In [ ]:
results_bundle["metrics"]["performance_p95_latency_ms"] = round(p95_robust, 3)
results_bundle["metrics"]["performance_median_latency_ms"] = round(median_robust, 3)

with open("/content/evaluation_results.json", "w") as f:
    json.dump(results_bundle, f, indent=2)

print("Updated evaluation_results.json with robust p95")

Updated evaluation_results.json with robust p95


In [ ]:
def clean(val):
    if val is None:
        return None
    if isinstance(val, float) and math.isnan(val):
        return None
    return val

aed_export = []
for _, row in gdf.iterrows():
    aed_export.append({
        "AED_ID": row["AED_ID"],
        "LATITUDE": float(row.geometry.y),
        "LONGITUDE": float(row.geometry.x),
        "BUILDING_NAME": clean(row["BUILDING_NAME"]),
        "AED_LOCATION_DESCRIPTION": clean(row["AED_LOCATION_DESCRIPTION"]),
        "AED_LOCATION_FLOOR_LEVEL": clean(row.get("AED_LOCATION_FLOOR_LEVEL")),
        "parsed_hours": row["parsed_hours"],
    })

with open("/content/aed_parsed_backend.json", "w") as f:
    json.dump(aed_export, f, indent=2)

In [ ]:
files.download("/content/aed_parsed.geojson")
files.download("/content/aed_parsed_backend.json")
files.download("/content/cached_scenarios.json")
files.download("/content/cached_routes.json")
files.download("/content/evaluation_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
check = gpd.read_file("/content/aed_parsed.geojson")
print(type(check['parsed_hours'].iloc[0]))
print(check['parsed_hours'].iloc[0])

<class 'str'>
{'status': 'parsed', 'confidence': 'high', 'schedule': {'MON': ('00:00', '23:59'), 'TUE': ('00:00', '23:59'), 'WED': ('00:00', '23:59'), 'THU': ('00:00', '23:59'), 'FRI': ('00:00', '23:59'), 'SAT': ('00:00', '23:59'), 'SUN': ('00:00', '23:59')}, 'has_remarks': False, 'remark_type': None, 'raw': 'Mon - Sun 00:00-23:59;'}


In [ ]:
with open("/content/aed_parsed_backend.json") as f:
    check2 = json.load(f)

print(type(check2[0]['parsed_hours']))
print(check2[0]['parsed_hours'])

<class 'dict'>
{'status': 'parsed', 'confidence': 'high', 'schedule': {'MON': ['00:00', '23:59'], 'TUE': ['00:00', '23:59'], 'WED': ['00:00', '23:59'], 'THU': ['00:00', '23:59'], 'FRI': ['00:00', '23:59'], 'SAT': ['00:00', '23:59'], 'SUN': ['00:00', '23:59']}, 'has_remarks': False, 'remark_type': None, 'raw': 'Mon - Sun 00:00-23:59;'}
